In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers "xformers<0.0.27" peft accelerate bitsandbytes
!pip install langchain
!pip install langchain-community
!pip install fitz
!pip install pymupdf
!pip install unstructured python-magic
!pip install faiss-gpu
!pip install transformers torch huggingface_hub
!pip install python-dotenv==1.0.0 streamlit==1.22.0 tiktoken==0.4.0
!pip install protobuf~=3.20
!pip install sentence-transformers
!pip install trl

In [ ]:
import torch
from trl import SFTTrainer
from transformers import TrainingArguments

from datasets import load_dataset


import logging
import numpy as np
import faiss
import os
import pandas as pd
import magic
import nltk
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.document_loaders import UnstructuredURLLoader
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.document_loaders import DirectoryLoader
from langchain.prompts import PromptTemplate
from langchain import PromptTemplate
from google.colab import userdata
from transformers import TextStreamer


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install triton
!pip uninstall xformers
!pip install xformers
from unsloth import FastLanguageModel
max_seq_length =  1024
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = True,
    random_state = 3407,
    use_rslora = True,
    loftq_config = None,
)

In [ ]:
alpaca_prompt = """Below is a query asked by a user , paired with additonal data that provides further context. Write a response that appropriately answers the query.
You must answer only from the given data and not make up anything. Be as detailed as possible. If the answer is not present in the data, just say "I don't know".

### Query:
{}

### Data:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

In [ ]:
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN #Adding EOS token as special token to vocabulary
        texts.append(text)
    return { "text" : texts, }
pass

In [ ]:
dataset = load_dataset("yahma/alpaca-cleaned", split = "train")
dataset = dataset.map(formatting_prompts_func, batched = True,)

In [ ]:
dataset

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 100,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# alpaca_prompt = from above
FastLanguageModel.for_inference(model) #Testing
inputs = tokenizer(
[
    alpaca_prompt.format(
        "Say the opposite of the data porvided.", # query
        "Hello", # data
        "", # output
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)

In [ ]:
model.save_pretrained("/content/drive/MyDrive/llama_model")
tokenizer.save_pretrained("/content/drive/MyDrive/llama3_8b/model")

In [ ]:
max_seq_length = 4096
dtype = None
load_in_4bit = True

if True:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "/content/drive/MyDrive/llama_model", # Loading the model using the saved path
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit, # loading in 4-bit to preserve memory and enhance speed

    )
    FastLanguageModel.for_inference(model)

In [ ]:
alpaca_prompt = """

### Query:
{}

### Data:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

In [ ]:
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN #Adding EOS token as special token to vocabulary
        texts.append(text)
    return { "text" : texts, }
pass

document loading

In [ ]:
documents = []
file_path = '/content/drive/MyDrive/heart_report.pdf'
loader = PyMuPDFLoader(file_path)
documents = loader.load()

In [ ]:
print(f"Number of documents loaded: {len(documents)}")

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200)

split_documents = text_splitter.split_documents(documents)
print(f"Number of chunks: {len(split_documents)}")

In [ ]:
embeddings = HuggingFaceEmbeddings()

In [ ]:
save_faiss = "/content/drive/MyDrive/faiss_store"

In [ ]:
vector_db = FAISS.from_documents(documents = split_documents, embedding = embeddings)
vector_db.save_local(save_faiss)

In [ ]:
vector_db = FAISS.load_local(save_faiss, embeddings, allow_dangerous_deserialization=True)

search and input


In [ ]:
user_input = input("Please enter your query: ")

In [ ]:
docs = vector_db.similarity_search_with_score(user_input, k = 5)
docs

In [ ]:
similar_documents = []
scores = []
for doc, score in docs:
  similar_documents.append(doc)
  scores.append(score)

similar_documents

In [ ]:
page_contents = [doc.page_content for doc in similar_documents]
metadata = [doc.metadata for doc in similar_documents]
#enumerate page content
print("Page Contents:")
for i, content in enumerate(page_contents):
    print(f"Document {i+1} Content:\n{content}\n")

# to get metadata
print("Metadata:")
for i, meta in enumerate(metadata):
    print(f"Document {i+1} Metadata:\n{meta}\n")

output ot inference

In [ ]:


    docs = vector_db.similarity_search_with_score(user_input, k=5)
    similar_documents = []
    scores = []
    for doc, score in docs:
        similar_documents.append(doc)
        scores.append(score)

    inputs = tokenizer(
        [
            alpaca_prompt.format(
                user_input,  # query
                page_contents,  # data
                "",  # output
            )
        ],
        return_tensors="pt"
    ).to("cuda")

    text_streamer = TextStreamer(tokenizer)
    outputs = model.generate(**inputs, streamer=text_streamer, max_new_tokens=4096)
    response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]